In [1]:
!pip install -U transformers sentence-transformers tf-keras

In [2]:
!pip install faiss-cpu

In [3]:
import faiss
print(faiss.__version__)

1.14.3


In [4]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from pathlib import Path

print("LOADING FULL SHARD")

df = pd.read_parquet(
    "hf://datasets/facebook/wiki_dpr/"
    "data/psgs_w100/"
    "nq/train-00000-of-00157.parquet"
)

print("Passages:", len(df))
print("Unique titles:", df["title"].nunique())

# Keep entire shard
kb = df.copy()

EXP_DIR = Path("experiments/rag_graphrag_full_shard")
EXP_DIR.mkdir(parents=True, exist_ok=True)

kb.to_parquet(
    EXP_DIR / "full_shard_original_passages.parquet",
    index=False
)

print("Saved!")
print("Titles:", kb["title"].nunique())
print("Passages:", len(kb))

LOADING FULL SHARD
Passages: 133856
Unique titles: 4352
Saved!
Titles: 4352
Passages: 133856


# NQ Benchmark preparation

In [6]:
# =========================
# NQ-Open Dataset Loading
# =========================

from datasets import load_dataset
import re

print("Loading NQ-Open validation set...")


try:
    nq = load_dataset(
        "nq_open",
        split="validation"
    )

    nq_df = pd.DataFrame(nq)

except Exception as e:
    print("Normal loading failed, switching to streaming mode.")
    print(e)

    nq_stream = load_dataset(
        "nq_open",
        split="validation",
        streaming=True
    )

    nq_examples = []

    for ex in nq_stream:
        nq_examples.append({
            "question": ex["question"],
            "answers": ex["answer"]
        })

    nq_df = pd.DataFrame(nq_examples)

# Standardize column names
if "answer" in nq_df.columns and "answers" not in nq_df.columns:
    nq_df = nq_df.rename(columns={"answer": "answers"})

# Ensure answers are always lists
def ensure_list(x):
    if isinstance(x, list):
        return x
    return [x]

nq_df["answers"] = nq_df["answers"].apply(ensure_list)

# Use first answer as main answer for convenience
nq_df["answer"] = nq_df["answers"].apply(
    lambda x: x[0] if len(x) > 0 else ""
)

# Text normalization function for later filtering / EM / F1
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("Loaded NQ examples:", len(nq_df))
print(nq_df.columns)
display(nq_df.head())

Loading NQ-Open validation set...


Using the latest cached version of the dataset since nq_open couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'nq_open' at /home/jovyan/.cache/huggingface/datasets/nq_open/nq_open/0.0.0/5dd9790a83002ad084ddeb7c420dc716852c6f28 (last modified on Sat Jun 13 12:52:52 2026).


Loaded NQ examples: 3610
Index(['question', 'answers', 'answer'], dtype='object')


,question,answers,answer
0,when was the last time anyone was on the moon,"[14 December 1972 UTC, December 1972]",14 December 1972 UTC
1,who wrote he ain't heavy he's my brother lyrics,"[Bobby Scott, Bob Russell]",Bobby Scott
2,how many seasons of the bastard executioner ar...,"[one, one season]",one
3,when did the eagles win last super bowl,[2017],2017
4,who won last year's ncaa women's basketball,[South Carolina],South Carolina


In [7]:
import os

QA_DIR = "qa_data"
os.makedirs(QA_DIR, exist_ok=True)

nq_df.to_parquet("qa_data/nq_open_validation.parquet", index=False)

print("Saved:", "qa_data/nq_open_validation.parquet")
print("Rows:", len(nq_df))

Saved: qa_data/nq_open_validation.parquet
Rows: 3610



# DPR Passage Embedding & FAISS Indexing
# (Single-Shard Wiki-DPR Corpus)


In [8]:
# =========================
# Dense DPR Retriever
# =========================

!pip install -q sentence-transformers faiss-gpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# DPR context encoder for passages
ctx_model = SentenceTransformer(
    "sentence-transformers/facebook-dpr-ctx_encoder-single-nq-base",
    device=device
)

texts = kb["text"].astype(str).tolist()
titles = kb["title"].astype(str).tolist()

print("Passages:", len(texts))

passage_embeddings = ctx_model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings shape:", passage_embeddings.shape)

dim = passage_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(passage_embeddings)

print("FAISS indexed passages:", index.ntotal)

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Passages: 133856


Batches:   0%|          | 0/1046 [00:00<?, ?it/s]

Embeddings shape: (133856, 768)
FAISS indexed passages: 133856


 DPR Context Encoder & FAISS Index Construction
- Encode Wikipedia passages.
- Build dense retrieval index.

In [9]:
# =========================
# DPR Question Encoder + Retrieval
# =========================

from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer

q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)

q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to(device)

q_encoder.eval()

def retrieve(question, top_k=5):
    inputs = q_tokenizer(
        question,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        q_emb = q_encoder(**inputs).pooler_output.cpu().numpy().astype("float32")

    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append({
            "score": float(score),
            "id": kb.iloc[idx]["id"],
            "title": kb.iloc[idx]["title"],
            "text": kb.iloc[idx]["text"]
        })

    return results

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Question Encoder
- Encode user questions
- Retrieve top-k relevant passages.

In [10]:
def answer_in_docs(answers, docs):
    docs_text = " ".join([d["text"] for d in docs])
    docs_text = normalize_text(docs_text)

    return any(
        normalize_text(ans) in docs_text
        for ans in answers
    )

def evaluate_hit_at_k(df, top_k=5):
    hits = 0

    for _, row in df.iterrows():
        docs = retrieve(row["question"], top_k=top_k)

        if answer_in_docs(row["answers"], docs):
            hits += 1

    return hits / len(df)


results = []

for k in [1, 5, 10, 20]:
    score = evaluate_hit_at_k(nq_df, top_k=k)

    results.append({
        "Dataset": "NQ-Open validation",
        "Corpus": "Full DPR shard 0",
        "Method": "DPR Dense Retrieval",
        "Metric": f"Hit@{k}",
        "Score": round(score * 100, 2)
    })

results_df = pd.DataFrame(results)
results_df

,Dataset,Corpus,Method,Metric,Score
0,NQ-Open validation,Full DPR shard 0,DPR Dense Retrieval,Hit@1,8.03
1,NQ-Open validation,Full DPR shard 0,DPR Dense Retrieval,Hit@5,18.09
2,NQ-Open validation,Full DPR shard 0,DPR Dense Retrieval,Hit@10,23.30
3,NQ-Open validation,Full DPR shard 0,DPR Dense Retrieval,Hit@20,28.59


Using a single Wiki-DPR shard as the retrieval corpus, the DPR retriever achieved Hit@1 = 8.03%, Hit@5 = 18.09%, Hit@10 = 23.30%, and Hit@20 = 28.59%. As expected, retrieval performance improves with larger values of k, indicating that the correct answer is more likely to appear among a broader set of retrieved passages. Although these scores are lower than those reported in the original DPR/RAG literature, the comparison is not direct because this experiment indexes only one shard of the Wikipedia corpus rather than the complete collection. Nevertheless, the results demonstrate that the retriever is capable of identifying relevant passages from a substantially reduced search space.

In [10]:
def answer_in_docs(answers, docs):
    docs_text = " ".join([d["text"] for d in docs])
    docs_text = normalize_text(docs_text)

    return any(
        normalize_text(ans) in docs_text
        for ans in answers
    )


for _ in range(10):
    row = nq_df.sample(1).iloc[0]

    docs = retrieve(row["question"], top_k=20)

    hit = answer_in_docs(row["answers"], docs)

    print("QUESTION:", row["question"])
    print("ANSWERS:", row["answers"])
    print("HIT:", hit)
    print("-" * 80)

QUESTION: first day collection of mission china assamese film
ANSWERS: ['₹ 39.97 lakh']
HIT: False
--------------------------------------------------------------------------------
QUESTION: who built the tower of london in 1066
ANSWERS: ['William the Conqueror']
HIT: True
--------------------------------------------------------------------------------
QUESTION: who is responsible for establishing local licensing forum
ANSWERS: ['unitary authorities', 'local authorities', 'district councils']
HIT: False
--------------------------------------------------------------------------------
QUESTION: who pays medical bills in great britain where does the money come from to pay these bills
ANSWERS: ['general taxation', 'taxes']
HIT: True
--------------------------------------------------------------------------------
QUESTION: who plays the woodsman in over the garden wall
ANSWERS: ['Christopher Lloyd']
HIT: False
--------------------------------------------------------------------------------
Q

Manual inspection of randomly sampled questions confirmed that retrieved passages frequently contained the expected answer when a hit was recorded, providing additional confidence in the correctness of the evaluation procedure.

To analyze ranking quality independently of corpus coverage, a retriever-answerable subset was constructed by retaining only questions whose answers appeared within the top-20 retrieved passages. On this subset, retrieval performance increased substantially, reaching 81.49% Hit@10 and 100% Hit@20. This indicates that once relevant evidence is present within the indexed corpus, the retriever is generally effective at ranking answer-containing passages.

# Flan T5 generator
--------------------

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import string
from collections import Counter

gen_name = "google/flan-t5-large"

gen_tokenizer = AutoTokenizer.from_pretrained(gen_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_name).to(device)

def generate_with_flan_t5(question, docs, top_k=5):
    docs = docs[:top_k]

    context = "\n\n".join([
        f"Title: {d['title']}\nPassage: {d['text']}"
        for d in docs
    ])

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    return gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()






def normalize_text(s):
    if not isinstance(s, str):
        s = str(s)
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = " ".join(s.split())
    return s

def exact_match_score(prediction, ground_truths):
    if isinstance(ground_truths, str):
        ground_truths = [ground_truths]
    pred = normalize_text(prediction)
    return float(any(pred == normalize_text(gt) for gt in ground_truths))

def f1_score_single(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(ground_truth).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def f1_multi(prediction, ground_truths):
    if isinstance(ground_truths, str):
        ground_truths = [ground_truths]
    return max(f1_score_single(prediction, gt) for gt in ground_truths)


def evaluate_flan_t5_prompt(
    benchmark_df,
    limit=50,
    top_k=5
):
    results = []

    subset = benchmark_df.head(limit)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        q = row["question"]
        gold = row["answers"]

        try:
            docs = retrieve(q, top_k=top_k)

            pred = generate_with_flan_t5(
                q,
                docs,
                top_k=top_k
            )

            em = exact_match_score(pred, gold)
            f1 = f1_multi(pred, gold)

            results.append({
                "question": q,
                "answers": gold,
                "predicted_answer": pred,
                "retrieved_titles": [d["title"] for d in docs],
                "EM": em,
                "F1": f1,
                "status": "ok"
            })

        except Exception as e:
            results.append({
                "question": q,
                "answers": gold,
                "predicted_answer": "",
                "retrieved_titles": [],
                "EM": 0.0,
                "F1": 0.0,
                "status": str(e)
            })

    result_df = pd.DataFrame(results)

    print("Examples:", len(result_df))
    print("EM:", round(result_df["EM"].mean() * 100, 2))
    print("F1:", round(result_df["F1"].mean() * 100, 2))

    return result_df








Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
flan_k1_nq = evaluate_flan_t5_prompt(
    nq_df,
    limit=1000,
    top_k=1
)

flan_k5_nq = evaluate_flan_t5_prompt(
    nq_df,
    limit=1000,
    top_k=5
)

flan_k10_nq = evaluate_flan_t5_prompt(
    nq_df,
    limit=1000,
    top_k=10
)

100%|██████████| 1000/1000 [03:34<00:00,  4.66it/s]


Examples: 1000
EM: 5.6
F1: 8.89


100%|██████████| 1000/1000 [04:49<00:00,  3.45it/s]


Examples: 1000
EM: 5.6
F1: 9.82


 38%|███▊      | 382/1000 [02:48<04:33,  2.26it/s]


KeyboardInterrupt: 

In [ ]:
check_df = flan_k10_nq  # or flan_k5_nq / flan_k1_nq

for i in range(5):
    row = check_df.iloc[i]

    print("QUESTION:", row["question"])
    print("GOLD:", row["answers"])
    print("PRED:", row["predicted_answer"])
    print("EM:", row["EM"], "F1:", row["F1"])
    print("RETRIEVED TITLES:", row["retrieved_titles"])
    print("-" * 100)

for i in range(5):
    q = nq_df.iloc[i]["question"]
    gold = nq_df.iloc[i]["answers"]

    docs = retrieve(q, top_k=10)
    pred = generate_with_flan_t5(q, docs, top_k=10)

    print("QUESTION:", q)
    print("GOLD:", gold)
    print("PRED:", pred)
    print("ANSWER IN DOCS:", answer_in_docs(gold, docs))

    for j, d in enumerate(docs[:3]):
        print(f"DOC {j+1}:", d["title"])
        print(d["text"][:300])

    print("=" * 100)

## Qualitative Analysis

To better understand the behavior of the retrieval-augmented QA pipeline, several examples from the NQ validation set were manually inspected. The analysis revealed that answer quality is strongly influenced by the quality of the retrieved passages. When relevant evidence was retrieved, the model was often able to generate correct answers. For example, the question "When was the last time anyone was on the moon?" was correctly answered as "December 1972" because the retrieved passages contained information about Apollo 17 and the final lunar mission.

## Error Analysis

Most incorrect predictions were associated with retrieval failures rather than generation failures. For example, for the question *"Who wrote He Ain't Heavy, He's My Brother?"*, the retrieved passages were unrelated to the song and its authors, causing the model to generate an incorrect answer. Similarly, for *"Who won last year's NCAA women's basketball?"*, the retrieved documents did not contain information about the target event, leading to an incorrect prediction. These examples suggest that the retrieval component is currently the primary bottleneck of the system. can be enhanced by using KG or modern retreiv system 


## Conclusion

The experimental results demonstrate the expected behavior of a retrieval-augmented generation pipeline. When relevant passages are successfully retrieved, FLAN-T5 is generally capable of producing accurate answers. However, when retrieval fails to surface supporting evidence, answer generation becomes unreliable. Therefore, overall system performance is largely constrained by retrieval quality. Future improvements could focus on using , a **hybrid retrieval** approaches to improve passage coverage and downstream question answering accuracy.

------------------------

 # Experiment 1: DPR-Retrieved Chunk Graph Evidence Augmentation (without graph retrieval)
 (without GRAPH RETRIEVAL use the DPR)

This section extends the baseline Dense Passage Retrieval (DPR) retrieval pipeline by incorporating graph-structured evidence extracted from retrieved passages. The implementation loads graph extraction results, matches retrieved chunks to graph records, extracts entity and relationship information, and constructs a hybrid retrieval context for answer generation using FLAN-T5.

The module also includes diagnostic utilities to evaluate the alignment between DPR-retrieved passages and graph extraction outputs.

In [12]:
# ============================================================
# HYBRID DPR + GRAPH EVIDENCE RAG + GRAPH MATCH DIAGNOSTIC
# ============================================================

import json
from pathlib import Path
from collections import defaultdict


# import llM extraction file that had with microsoft paper -> Graph - RAG

GRAPH_JSONL = Path("graph_extraction_checkpoint/graph_extraction_results.jsonl")

if not GRAPH_JSONL.exists():
    raise FileNotFoundError(
        f"Could not find {GRAPH_JSONL}. "
        "Upload/copy graph_extraction_results.jsonl into graph_extraction_checkpoint/ "
        "or change GRAPH_JSONL to the correct path."
    )

# ------------------------------------------------------------
# 1. Load graph extraction file
# ------------------------------------------------------------

graph_by_chunk = {}                              #chunk id --> graph record                           
graph_by_title = defaultdict(list)               #title -->list of graph records       

with open(GRAPH_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)
        cid = str(row.get("chunk_id", "")).strip()

        if not cid:
            continue

        graph_by_chunk[cid] = row
        graph_by_title[str(row.get("title", "")).strip()].append(row)

print("Loaded graph extraction records:", len(graph_by_chunk))
print("Unique graph titles:", len(graph_by_title))


# ------------------------------------------------------------
# 2. ID helper for matching DPR docs to graph records
# ------------------------------------------------------------

def _id_variants(x):
    x = str(x).strip()
    variants = {x}

    try:
        variants.add(str(int(float(x))))
    except Exception:
        pass

    return variants


def get_graph_record_for_doc(doc, use_title_fallback=False):
    """
    Match a DPR-retrieved document to a graph extraction record.
    First tries doc['id'] against graph chunk_id.
    Optional title fallback can be enabled.
    """

    doc_id = doc.get("id", "")

    for v in _id_variants(doc_id):
        if v in graph_by_chunk:
            return graph_by_chunk[v], "id"

    if use_title_fallback:
        title = str(doc.get("title", "")).strip()
        candidates = graph_by_title.get(title, [])
        if len(candidates) > 0:
            return candidates[0], "title"

    return None, "none"


# ------------------------------------------------------------
# 3. Diagnostic: can DPR retrieved docs match graph chunks?
# ------------------------------------------------------------

def graph_match_diagnostic(eval_df, sample_size=100, top_k=5, use_title_fallback=False):
    sample = eval_df.head(sample_size)

    total_docs = 0                                    #-> how many documetns were checked 
    matched_docs = 0                                  # how many had graph record
    match_types = defaultdict(int)                    #  #how many matched by id

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Graph match diagnostic"):
        docs = retrieve(row["question"], top_k=top_k)

        for d in docs:
            total_docs += 1

            rec, match_type = get_graph_record_for_doc(
                d,
                use_title_fallback=use_title_fallback
            )

            match_types[match_type] += 1

            if rec is not None:
                matched_docs += 1

    print("DPR docs checked:", total_docs)
    print("Matched graph records:", matched_docs)
    print("Match rate:", round(matched_docs / max(1, total_docs) * 100, 2), "%")
    print("Match types:", dict(match_types))


# ------------------------------------------------------------
# 4. Build graph evidence from DPR-retrieved chunk IDs
# ------------------------------------------------------------

def graph_evidence_from_chunks(chunk_ids, max_relations=12, max_entities=12):
    evidence_parts = []
    seen_relations = set()                # avoid duplicate relations
    seen_entities = set()                  # avoid duplicate entities

    evidence_parts.append("Graph evidence from retrieved passages:")

    for cid in chunk_ids:
        cid = str(cid).strip()

        rec = None

        for v in _id_variants(cid):
            if v in graph_by_chunk:
                rec = graph_by_chunk[v]
                break                                             # matching grpah data for the chunk

        if rec is None:
            continue

        title = rec.get("title", "")

        # Entities
        for e in rec.get("entities", []):
            name = str(e.get("entity_name", "")).strip()
            desc = str(e.get("entity_description", "")).strip()

            if not name or name.upper() in {"OTHER", "PERSON", "GROUP", "DATE", "CONCEPT"}:
                continue

            key = name.lower()

            if key in seen_entities:
                continue

            seen_entities.add(key)

            if len(seen_entities) <= max_entities:                        #prevent prompt becoing too long
                evidence_parts.append(
                    f"Entity: {name}\n"
                    f"Description: {desc}\n"
                    f"Source title: {title}"
                )

        # Relationships
        for r in rec.get("relationships", []):
            source = str(r.get("source_entity", "")).strip()                             # source entity from the extracted JSON 
            target = str(r.get("target_entity", "")).strip()                              # target entity
            desc = str(r.get("relationship_description", "")).strip()                    # the description of the relationship
            ev = str(r.get("evidence", "")).strip()                                      # evidence 
            strength = r.get("relationship_strength", 0) or 0                             # strngth

            if not source or not target:
                continue

            key = (source.lower(), target.lower(), desc.lower())

            if key in seen_relations:
                continue

            seen_relations.add(key)

            if len(seen_relations) <= max_relations:
                evidence_parts.append(
                    f"Relation: {source} -> {target}\n"
                    f"Description: {desc}\n"
                    f"Evidence: {ev}\n"
                    f"Strength: {strength}\n"
                    f"Source title: {title}"
                )

    return "\n\n".join(evidence_parts)


# ------------------------------------------------------------
# 5. Build hybrid context for FLAN-T5
# ------------------------------------------------------------

def build_hybrid_context(question, retrieved_docs, max_passages=5, max_chars=4000):
    parts = []

    parts.append("Question: " + str(question))                    #Question \ query
    parts.append("\nRetrieved DPR passages:")                     #retreived DPR PASSAGES

    chunk_ids = []

    for doc in retrieved_docs[:max_passages]:
        cid = str(doc.get("id", "")).strip()
        title = doc.get("title", "")
        text = doc.get("text", "")

        chunk_ids.append(cid)               # apüend all ids of the retreived chunks

        parts.append(
            f"Title: {title}\n"
            f"Chunk id: {cid}\n"
            f"Text: {str(text)[:900]}"
        )

    graph_text = graph_evidence_from_chunks(
        chunk_ids,
        max_relations=10,
        max_entities=8
    )

    parts.append("\n" + graph_text)

    return "\n\n".join(parts)[:max_chars]


# ------------------------------------------------------------
# 6. Hybrid answer function
# ------------------------------------------------------------

def hybrid_graph_rag_answer(question, top_k=5):
    retrieved_docs = retrieve(question, top_k=top_k)

    context = build_hybrid_context(
        question=question,
        retrieved_docs=retrieved_docs,
        max_passages=top_k
    )

    answer = generate_with_flan_t5(question, context)

    return answer, context, retrieved_docs


print("Hybrid Graph-RAG utilities ready.")



##########################################################################





# ============================================================
# Hybrid DPR + Graph Evidence RAG Evaluation
# ============================================================


# REMOVE THE STOP WORDS FROM QUERY ONLY  --> ONLY IN THE MATCHING STEP 

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "at", "to", "for", "by",
    "is", "are", "was", "were", "be", "been", "being",
    "who", "what", "when", "where", "why", "how",
    "did", "do", "does", "there", "last", "many", "about",
    "name", "called", "known",
    "person", "group", "date", "concept", "other",
    "thing", "place", "time", "year", "years",
    "first", "also", "much"
}

def query_terms(question):
    toks = re.findall(r"[a-z0-9]+", str(question).lower())
    return {t for t in toks if len(t) > 2 and t not in STOPWORDS}                    # reomve short and stope wros


def build_graph_evidence(docs, question=None, max_entities=2, max_relations=2):
    q_terms = query_terms(question) if question is not None else set()                        # extarct key words form the question

    entity_scored = []
    relation_scored = []
    seen_entities = set()
    seen_relations = set()

    for d in docs:                                                                                   #for each RETREIVED DPR
        rec, _ = get_graph_record_for_doc(d, use_title_fallback=True)

        if rec is None:
            continue

        # -------------------------
        # Relevant entity evidence
        # -------------------------
        for e in rec.get("entities", []):
            name = str(e.get("entity_name", "")).strip()
            etype = str(e.get("entity_type", "")).strip()
            desc = str(e.get("entity_description", "")).strip()

            if not name or not desc:
                continue
  
            text = f"{name} {etype} {desc}".lower()                                           #EX-->  MAX MARTIN PERSON Max Martin is a producer who worked on the album                             
            ent_terms = set(re.findall(r"[a-z0-9]+", text))
            overlap = len(q_terms & ent_terms)                                               #calculate manual overlap without tokenizer

            if question is not None and overlap == 0:
                continue

            key = name.lower()

            if key in seen_entities:
                continue

            seen_entities.add(key)

            score = overlap * 10                            # high overlap --> more relavant entity between quey and retrieved

            entity_scored.append((
                score,
                f"Entity: {name}. Description: {desc}"
            ))

        # -------------------------
        # Relevant relation evidence
        # -------------------------
        for r in rec.get("relationships", []):
            src = str(r.get("source_entity", "")).strip()
            tgt = str(r.get("target_entity", "")).strip()
            desc = str(r.get("relationship_description", "")).strip()
            ev = str(r.get("evidence", "")).strip()
            strength = r.get("relationship_strength", 0) or 0

            if not src or not tgt:
                continue

            text = f"{src} {tgt} {desc} {ev}".lower()
            rel_terms = set(re.findall(r"[a-z0-9]+", text))
            overlap = len(q_terms & rel_terms)

            if question is not None and overlap == 0:
                continue

            key = (src.lower(), tgt.lower(), desc.lower())

            if key in seen_relations:
                continue

            seen_relations.add(key)

            score = overlap * 10 + strength

            relation_scored.append((
                score,
                f"Relation: {src} -> {tgt}. Description: {desc}. Evidence: {ev}"
            ))

    entity_scored = sorted(entity_scored, reverse=True)
    relation_scored = sorted(relation_scored, reverse=True)

    evidence_parts = []

    for _, text in entity_scored[:max_entities]:
        evidence_parts.append(text)

    for _, text in relation_scored[:max_relations]:
        evidence_parts.append(text)

    return "\n".join(evidence_parts)




def generate_with_flan_t5_hybrid(question, docs, top_k=5, max_relations=3):
    # Same baseline context style used by generate_with_flan_t5
    context = "\n\n".join([
        f"Title: {d['title']}\nPassage: {d['text']}"
        for d in docs[:top_k]
    ])

    graph_context = build_graph_evidence(
        docs,
        question=question,
        max_relations=max_relations
    ).strip()

    # Only add graph evidence if useful evidence exists
    if graph_context:
        hybrid_context = (
            context
            + "\n\nAdditional relevant graph evidence:\n"
            + graph_context
        )
    else:
        hybrid_context = context

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{hybrid_context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    return gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip(), graph_context


def evaluate_flan_t5_hybrid_prompt(
    benchmark_df,
    limit=1000,
    top_k=5,
    max_relations=3
):
    results = []

    subset = benchmark_df.head(limit)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        q = row["question"]
        gold = row["answers"]

        try:
            docs = retrieve(q, top_k=top_k)

            # Original baseline function
            baseline_pred = generate_with_flan_t5(
                q,
                docs,
                top_k=top_k
            )

            # Hybrid function
            hybrid_pred, graph_context = generate_with_flan_t5_hybrid(
                q,
                docs,
                top_k=top_k,
                max_relations=max_relations
            )

            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": baseline_pred,
                "hybrid_pred": hybrid_pred,
                "retrieved_titles": [d["title"] for d in docs],
                "graph_context": graph_context,
                "graph_evidence_chars": len(graph_context),
                "baseline_EM": exact_match_score(baseline_pred, gold),
                "hybrid_EM": exact_match_score(hybrid_pred, gold),
                "baseline_F1": f1_multi(baseline_pred, gold),
                "hybrid_F1": f1_multi(hybrid_pred, gold),
                "status": "ok"
            })

        except Exception as e:
            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": "",
                "hybrid_pred": "",
                "retrieved_titles": [],
                "graph_context": "",
                "graph_evidence_chars": 0,
                "baseline_EM": 0.0,
                "hybrid_EM": 0.0,
                "baseline_F1": 0.0,
                "hybrid_F1": 0.0,
                "status": str(e)
            })

    result_df = pd.DataFrame(results)

    baseline_em = result_df["baseline_EM"].mean() * 100
    hybrid_em = result_df["hybrid_EM"].mean() * 100
    baseline_f1 = result_df["baseline_F1"].mean() * 100
    hybrid_f1 = result_df["hybrid_F1"].mean() * 100
    graph_available = (result_df["graph_evidence_chars"] > 0).mean() * 100

    print("========== RESULTS ==========")
    print("Examples:", len(result_df))
    print("Top-k:", top_k)
    print("Max graph relations:", max_relations)

    print("\nBaseline EM:", baseline_em)
    print("Hybrid EM:", hybrid_em)
    print("EM improvement:", hybrid_em - baseline_em)

    print("\nBaseline F1:", baseline_f1)
    print("Hybrid F1:", hybrid_f1)
    print("F1 improvement:", hybrid_f1 - baseline_f1)

    print("\nGraph evidence available:", graph_available)

    return result_df




hybrid_test_100 = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=100,
    top_k=5,
    max_relations=3
)






hybrid_test_100 = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=100,
    top_k=5,
    max_relations=3
)






Loaded graph extraction records: 17315
Unique graph titles: 3535
Hybrid Graph-RAG utilities ready.


100%|██████████| 100/100 [01:04<00:00,  1.56it/s]


========== RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 3

Baseline EM: 4.0
Hybrid EM: 6.0
EM improvement: 2.0

Baseline F1: 8.066666666666666
Hybrid F1: 10.535714285714285
F1 improvement: 2.4690476190476183

Graph evidence available: 83.0


100%|██████████| 100/100 [01:02<00:00,  1.61it/s]

========== RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 3

Baseline EM: 4.0
Hybrid EM: 6.0
EM improvement: 2.0

Baseline F1: 8.066666666666666
Hybrid F1: 10.535714285714285
F1 improvement: 2.4690476190476183

Graph evidence available: 83.0


In [15]:
hybrid_k5_nq_improved = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=1000,
    top_k=5,
    max_relations=3
)



hybrid_k5_nq_improved = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=1000,
    top_k=10,
    max_relations=3
)


hybrid_k5_nq_improved = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=1000,
    top_k=5,
    max_relations=3
)



100%|██████████| 1000/1000 [09:23<00:00,  1.77it/s]


========== RESULTS ==========
Examples: 1000
Top-k: 5
Max graph relations: 3

Baseline EM: 5.6000000000000005
Hybrid EM: 6.4
EM improvement: 0.7999999999999998

Baseline F1: 9.815305250305249
Hybrid F1: 10.375384615384615
F1 improvement: 0.5600793650793658

Graph evidence available: 83.7


100%|██████████| 1000/1000 [15:00<00:00,  1.11it/s]


========== RESULTS ==========
Examples: 1000
Top-k: 10
Max graph relations: 3

Baseline EM: 7.8
Hybrid EM: 7.3
EM improvement: -0.5

Baseline F1: 12.322882395382395
Hybrid F1: 11.660514485514485
F1 improvement: -0.6623679098679105

Graph evidence available: 91.4


100%|██████████| 1000/1000 [09:23<00:00,  1.77it/s]

========== RESULTS ==========
Examples: 1000
Top-k: 5
Max graph relations: 3

Baseline EM: 5.6000000000000005
Hybrid EM: 6.4
EM improvement: 0.7999999999999998

Baseline F1: 9.815305250305249
Hybrid F1: 10.375384615384615
F1 improvement: 0.5600793650793658

Graph evidence available: 83.7


In [40]:
hybrid_k5_nq_improved = evaluate_flan_t5_hybrid_prompt(
    nq_df,
    limit=1000,
    top_k=10,
    max_relations=3
)





100%|██████████| 1000/1000 [14:55<00:00,  1.12it/s]

========== RESULTS ==========
Examples: 1000
Top-k: 10
Max graph relations: 3

Baseline EM: 7.8
Hybrid EM: 7.3999999999999995
EM improvement: -0.40000000000000036

Baseline F1: 12.322882395382395
Hybrid F1: 11.296798756798756
F1 improvement: -1.0260836385836392

Graph evidence available: 88.4


this run is wiht the new biuld_graph_evidence


# Experiment 2 --> Implicit Entity Relation Graph Retrieval from LLM-Extracted JSON  (Without Networkx)

------

In [15]:
# ============================================================
# GRAPH-ONLY RETRIEVAL + TRUE HYBRID DPR + GRAPH RETRIEVAL
# ============================================================

def normalize_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def query_terms(text):
    toks = re.findall(r"[a-z0-9]+", normalize_text(text))
    return {t for t in toks if len(t) > 2 and t not in STOPWORDS}

# -----------------------------
# 2. Build graph entity index
# -----------------------------
graph_by_entity = defaultdict(list)

for cid, rec in graph_by_chunk.items():
    for e in rec.get("entities", []):
        name = e.get("entity_name", "")
        norm_name = normalize_text(name)
        if norm_name:
            graph_by_entity[norm_name].append(rec)

print("Graph entity index size:", len(graph_by_entity))

# -----------------------------
# 3. Match question to graph nodes
# -----------------------------
def match_query_entities(question, top_k_entities=5):
    q_terms = query_terms(question)
    matches = []

    for entity_name, records in graph_by_entity.items():
        e_terms = query_terms(entity_name)

        if not e_terms:
            continue

        overlap = len(q_terms & e_terms)

        # also allow phrase containment
        phrase_bonus = 2 if entity_name in normalize_text(question) else 0

        score = overlap + phrase_bonus

        if score > 0:
            matches.append((score, entity_name, records))

    matches = sorted(matches, key=lambda x: x[0], reverse=True)
    return matches[:top_k_entities]

# -----------------------------
# 4. Graph-only retrieval
# -----------------------------
def graph_retrieve(question, top_k_entities=5, max_relations=10):
    matched_entities = match_query_entities(
        question,
        top_k_entities=top_k_entities
    )

    evidence = []
    seen = set()

    for entity_score, entity_name, records in matched_entities:
        for rec in records:
            for r in rec.get("relationships", []):
                src = str(r.get("source_entity", "")).strip()
                tgt = str(r.get("target_entity", "")).strip()
                desc = str(r.get("relationship_description", "")).strip()
                ev = str(r.get("evidence", "")).strip()
                strength = r.get("relationship_strength", 0) or 0

                if not src or not tgt:
                    continue

                norm_src = normalize_text(src)
                norm_tgt = normalize_text(tgt)

                if entity_name not in {norm_src, norm_tgt}:
                    continue

                key = (norm_src, norm_tgt, normalize_text(desc))
                if key in seen:
                    continue

                seen.add(key)

                score = entity_score * 10 + strength

                evidence.append({
                    "score": score,
                    "source": src,
                    "target": tgt,
                    "description": desc,
                    "evidence": ev,
                    "strength": strength,
                    "chunk_id": rec.get("chunk_id", ""),
                    "title": rec.get("title", "")
                })

    evidence = sorted(evidence, key=lambda x: x["score"], reverse=True)
    return evidence[:max_relations]

# -----------------------------
# 5. Build graph context
# -----------------------------
def build_graph_retrieval_context(question, max_relations=10):
    graph_results = graph_retrieve(
        question,
        top_k_entities=5,
        max_relations=max_relations
    )

    if not graph_results:
        return ""

    parts = ["Graph-retrieved evidence:"]

    for item in graph_results:
        parts.append(
            f"Relation: {item['source']} -> {item['target']}\n"
            f"Description: {item['description']}\n"
            f"Evidence: {item['evidence']}\n"
            f"Source title: {item['title']}"
        )

    return "\n\n".join(parts)

# -----------------------------
# 6. Graph-only RAG answer
# -----------------------------
def graph_only_rag_answer(question, max_relations=10):
    graph_context = build_graph_retrieval_context(
        question,
        max_relations=max_relations
    )

    if not graph_context:
        graph_context = "No graph evidence found."

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{graph_context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    return gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip(), graph_context

# -----------------------------
# 7. True hybrid: DPR + independent graph retrieval
# -----------------------------
def true_hybrid_rag_answer(question, top_k=5, max_relations=10):
    docs = retrieve(question, top_k=top_k)

    dpr_context = "\n\n".join([
        f"Title: {d['title']}\nPassage: {d['text']}"
        for d in docs
    ])

    graph_context = build_graph_retrieval_context(
        question,
        max_relations=max_relations
    )

    hybrid_context = dpr_context

    if graph_context:
        hybrid_context += "\n\nAdditional graph-retrieved evidence:\n"
        hybrid_context += graph_context

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{hybrid_context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    answer = gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return answer, hybrid_context, docs, graph_context

def evaluate_true_hybrid(
    benchmark_df,
    limit=100,
    top_k=5,
    max_relations=10
):
    results = []
    subset = benchmark_df.head(limit)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        q = row["question"]
        gold = row["answers"]

        try:
            docs = retrieve(q, top_k=top_k)

            baseline_pred = generate_with_flan_t5(
                q,
                docs,
                top_k=top_k
            )

            graph_pred, graph_context = graph_only_rag_answer(
                q,
                max_relations=max_relations
            )

            hybrid_pred, hybrid_context, _, graph_context = true_hybrid_rag_answer(
                q,
                top_k=top_k,
                max_relations=max_relations
            )

            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": baseline_pred,
                "graph_pred": graph_pred,
                "true_hybrid_pred": hybrid_pred,
                "graph_context": graph_context,
                "graph_evidence_chars": len(graph_context),
                "baseline_EM": exact_match_score(baseline_pred, gold),
                "graph_EM": exact_match_score(graph_pred, gold),
                "true_hybrid_EM": exact_match_score(hybrid_pred, gold),
                "baseline_F1": f1_multi(baseline_pred, gold),
                "graph_F1": f1_multi(graph_pred, gold),
                "true_hybrid_F1": f1_multi(hybrid_pred, gold),
                "status": "ok"
            })

        except Exception as e:
            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": "",
                "graph_pred": "",
                "true_hybrid_pred": "",
                "graph_context": "",
                "graph_evidence_chars": 0,
                "baseline_EM": 0.0,
                "graph_EM": 0.0,
                "true_hybrid_EM": 0.0,
                "baseline_F1": 0.0,
                "graph_F1": 0.0,
                "true_hybrid_F1": 0.0,
                "status": str(e)
            })

    result_df = pd.DataFrame(results)

    print("========== RESULTS ==========")
    print("Examples:", len(result_df))
    print("Top-k:", top_k)
    print("Max graph relations:", max_relations)

    print("\nBaseline EM:", result_df["baseline_EM"].mean() * 100)
    print("Graph-only EM:", result_df["graph_EM"].mean() * 100)
    print("True hybrid EM:", result_df["true_hybrid_EM"].mean() * 100)

    print("\nBaseline F1:", result_df["baseline_F1"].mean() * 100)
    print("Graph-only F1:", result_df["graph_F1"].mean() * 100)
    print("True hybrid F1:", result_df["true_hybrid_F1"].mean() * 100)

    print("\nGraph evidence available:",
          (result_df["graph_evidence_chars"] > 0).mean() * 100)

    return result_df

Graph entity index size: 114664


In [19]:
true_hybrid_test_100 = evaluate_true_hybrid(
    nq_df,
    limit=100,
    top_k=5,
    max_relations=10
)

100%|██████████| 100/100 [03:32<00:00,  2.13s/it]

========== RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 10

Baseline EM: 4.0
Graph-only EM: 3.0
True hybrid EM: 7.000000000000001

Baseline F1: 8.466666666666667
Graph-only F1: 7.022222222222223
True hybrid F1: 11.15

Graph evidence available: 100.0


In [20]:
evaluate_true_hybrid(
    nq_df,
    limit=1000,
    top_k=5,
    max_relations=10
)

100%|██████████| 1000/1000 [35:51<00:00,  2.15s/it]

========== RESULTS ==========
Examples: 1000
Top-k: 5
Max graph relations: 10

Baseline EM: 5.5
Graph-only EM: 4.7
True hybrid EM: 6.6000000000000005

Baseline F1: 10.50123839123839
Graph-only F1: 9.290324675324674
True hybrid F1: 11.853102453102453

Graph evidence available: 100.0


,question,answers,baseline_pred,graph_pred,true_hybrid_pred,graph_context,graph_evidence_chars,baseline_EM,graph_EM,true_hybrid_EM,baseline_F1,graph_F1,true_hybrid_F1,status
0,when was the last time anyone was on the moon,"[14 December 1972 UTC, December 1972]",December 1972,Apollo 11,December 1972,Graph-retrieved evidence:\n\nRelation: TIME ->...,2457,1.0,0.0,1.0,1.0,0.0,1.0,ok
1,who wrote he ain't heavy he's my brother lyrics,"[Bobby Scott, Bob Russell]",Don Siegel,BELL HOOKS,BELL HOOKS,Graph-retrieved evidence:\n\nRelation: BROTHER...,2763,0.0,0.0,0.0,0.0,0.0,0.0,ok
2,how many seasons of the bastard executioner ar...,"[one, one season]",five seasons,2,five,Graph-retrieved evidence:\n\nRelation: SEASONS...,1706,0.0,0.0,0.0,0.0,0.0,0.0,ok
3,when did the eagles win last super bowl,[2017],Super Bowl XV,1967,1938,Graph-retrieved evidence:\n\nRelation: SUPER B...,2628,0.0,0.0,0.0,0.0,0.0,0.0,ok
4,who won last year's ncaa women's basketball,[South Carolina],the Soviet Union,Michigan State,Michigan Wolverines,Graph-retrieved evidence:\n\nRelation: WOMEN -...,2476,0.0,0.0,0.0,0.0,0.0,0.0,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,number of degree of freedom for plane mechanism,"[six, two]",three,4,three,Graph-retrieved evidence:\n\nRelation: ANGLE -...,2449,0.0,0.0,0.0,0.0,0.0,0.0,ok
996,name of volcano that erupted in iceland in 2010,[Eyjafjallajökull],Berarbunga volcano,Iceland,Bárarbunga,Graph-retrieved evidence:\n\nRelation: ARCTIC ...,2352,0.0,0.0,0.0,0.0,0.0,0.0,ok
997,where does summer of the monkeys take place,"[Oklahoma, Missouri]",Cuba,New York City,Ketchum,Graph-retrieved evidence:\n\nRelation: MONKEYS...,2098,0.0,0.0,0.0,0.0,0.0,0.0,ok
998,who played young monica in love and basketball,[Kyla Pratt],Courtney Love,Kristen Joy Schaal,Courtney Love,Graph-retrieved evidence:\n\nRelation: YOUNG -...,2449,0.0,0.0,0.0,0.0,0.0,0.0,ok


In [16]:
true_hybrid_full = evaluate_true_hybrid(
    nq_df,
    limit=len(nq_df),
    top_k=5,
    max_relations=10
)

100%|██████████| 3610/3610 [2:13:18<00:00,  2.22s/it]  

========== RESULTS ==========
Examples: 3610
Top-k: 5
Max graph relations: 10

Baseline EM: 5.8171745152354575
Graph-only EM: 4.238227146814404
True hybrid EM: 6.260387811634349

Baseline F1: 10.582810428446436
Graph-only F1: 7.9856239482278255
True hybrid F1: 10.843333817856342

Graph evidence available: 100.0


In [16]:
true_hybrid_full_3 = evaluate_true_hybrid(
    nq_df,
    limit=len(nq_df),
    top_k=5,
    max_relations=3
)



100%|██████████| 3610/3610 [2:04:13<00:00,  2.06s/it]  

========== RESULTS ==========
Examples: 3610
Top-k: 5
Max graph relations: 3

Baseline EM: 5.8171745152354575
Graph-only EM: 5.096952908587258
True hybrid EM: 6.232686980609419

Baseline F1: 10.582810428446436
Graph-only F1: 9.35056863727224
True hybrid F1: 11.068845273732798

Graph evidence available: 100.0


In [19]:
true_hybrid_full_5 = evaluate_true_hybrid(
    nq_df,
    limit=len(nq_df),
    top_k=5,
    max_relations=5
)

100%|██████████| 3610/3610 [2:14:08<00:00,  2.23s/it]  

========== RESULTS ==========
Examples: 3610
Top-k: 5
Max graph relations: 5

Baseline EM: 5.8171745152354575
Graph-only EM: 4.404432132963989
True hybrid EM: 6.509695290858726

Baseline F1: 10.582810428446436
Graph-only F1: 8.602954755309327
True hybrid F1: 11.44054139636586

Graph evidence available: 100.0


# Final Experiment: Explicit Graph-RAG with Source-Chunk Evidence Expansion with NetworkX

In [13]:
# ============================================================
# FINAL EXPERIMENT: EXPLICIT GRAPH + CHUNK-CONTEXT GRAPH RETRIEVAL
# ============================================================


try:
    import networkx as nx
except ImportError:
    !pip install networkx
    import networkx as nx


# ============================================================
# 1. Normalization helpers
# ============================================================

def normalize_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def graph_terms(text):
    toks = re.findall(r"[a-z0-9]+", normalize_text(text))
    return {t for t in toks if len(t) > 2 and t not in STOPWORDS}


# ============================================================
# 2. Build explicit graph from graph_by_chunk
# ============================================================

G = nx.MultiDiGraph()
norm_to_node = {}

def get_or_create_node(name, entity_type="", description=""):
    name = str(name).strip()
    norm = normalize_text(name)

    if not norm:
        return None

    if norm in norm_to_node:
        node = norm_to_node[norm]
    else:
        node = name
        norm_to_node[norm] = node
        G.add_node(
            node,
            norm=norm,
            entity_type=entity_type,
            description=description
        )

    if entity_type and not G.nodes[node].get("entity_type"):
        G.nodes[node]["entity_type"] = entity_type

    if description and not G.nodes[node].get("description"):
        G.nodes[node]["description"] = description

    return node


# Add entity nodes
for cid, rec in graph_by_chunk.items():
    for e in rec.get("entities", []):
        get_or_create_node(
            e.get("entity_name", ""),
            e.get("entity_type", ""),
            e.get("entity_description", "")
        )


# Add relationship edges
for cid, rec in graph_by_chunk.items():
    title = rec.get("title", "")

    for r in rec.get("relationships", []):
        src = str(r.get("source_entity", "")).strip()
        tgt = str(r.get("target_entity", "")).strip()

        if not src or not tgt:
            continue

        src_node = get_or_create_node(src)
        tgt_node = get_or_create_node(tgt)

        if src_node is None or tgt_node is None:
            continue

        G.add_edge(
            src_node,
            tgt_node,
            description=str(r.get("relationship_description", "")).strip(),
            evidence=str(r.get("evidence", "")).strip(),
            strength=r.get("relationship_strength", 0) or 0,
            chunk_id=cid,
            title=title
        )

print("Explicit graph built.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


# ============================================================
# 3. Build chunk_id -> edges index
# This restores chunk-level context during graph retrieval
# ============================================================

edges_by_chunk = defaultdict(list)

for src, tgt, data in G.edges(data=True):
    cid = data.get("chunk_id", "")
    if cid:
        edges_by_chunk[cid].append((src, tgt, data))

print("Chunks with graph edges:", len(edges_by_chunk))


# ============================================================
# 4. Match question to graph nodes
# ============================================================

def is_noisy_node(node):
    norm = normalize_text(node)

    if not norm:
        return True

    if norm.upper() in {"OTHER", "PERSON", "GROUP", "DATE", "CONCEPT"}:
        return True

    if len(norm) <= 2:
        return True

    return False


def node_text_for_matching(node):
    data = G.nodes[node]

    return " ".join([
        str(node),
        str(data.get("entity_type", "")),
        str(data.get("description", ""))
    ])


def match_query_graph_nodes(question, top_k_entities=5):
    q_terms = graph_terms(question)
    q_norm = normalize_text(question)

    matches = []

    for node in G.nodes:
        if is_noisy_node(node):
            continue

        node_norm = normalize_text(node)
        node_terms = graph_terms(node_text_for_matching(node))

        if not node_terms:
            continue

        overlap = len(q_terms & node_terms)
        phrase_bonus = 4 if node_norm and node_norm in q_norm else 0
        specificity_bonus = min(len(node_norm.split()), 4) * 0.25

        score = overlap * 2 + phrase_bonus + specificity_bonus

        if score > 0:
            matches.append((score, node))

    matches = sorted(matches, key=lambda x: x[0], reverse=True)
    return matches[:top_k_entities]


# ============================================================
# 5. Graph retrieval with chunk-context expansion
# ============================================================

def graph_retrieve_explicit_with_chunk_context(
    question,
    top_k_entities=5,
    max_relations=3,
    expand_chunks=True
):
    q_terms = graph_terms(question)

    matched_nodes = match_query_graph_nodes(
        question,
        top_k_entities=top_k_entities
    )

    candidate_edges = []
    seen_edges = set()

    for entity_score, node in matched_nodes:
        direct_edges = []

        # Outgoing edges
        for _, tgt, key, data in G.out_edges(node, keys=True, data=True):
            direct_edges.append((node, tgt, data, entity_score))

        # Incoming edges
        for src, _, key, data in G.in_edges(node, keys=True, data=True):
            direct_edges.append((src, node, data, entity_score))

        for src, tgt, data, base_score in direct_edges:
            cid = data.get("chunk_id", "")

            edge_id = (
                normalize_text(src),
                normalize_text(tgt),
                normalize_text(data.get("description", "")),
                cid
            )

            if edge_id not in seen_edges:
                seen_edges.add(edge_id)
                candidate_edges.append((src, tgt, data, base_score))

            # Add other edges from the same chunk
            if expand_chunks and cid in edges_by_chunk:
                for c_src, c_tgt, c_data in edges_by_chunk[cid]:
                    c_edge_id = (
                        normalize_text(c_src),
                        normalize_text(c_tgt),
                        normalize_text(c_data.get("description", "")),
                        cid
                    )

                    if c_edge_id in seen_edges:
                        continue

                    seen_edges.add(c_edge_id)

                    # Lower score because this edge comes from chunk context
                    candidate_edges.append(
                        (c_src, c_tgt, c_data, base_score * 0.7)
                    )

    scored = []

    for src, tgt, data, base_score in candidate_edges:
        desc = str(data.get("description", "")).strip()
        ev = str(data.get("evidence", "")).strip()
        title = str(data.get("title", "")).strip()
        strength = data.get("strength", 0) or 0

        relation_text = f"{src} {tgt} {desc} {ev} {title}"
        overlap = len(q_terms & graph_terms(relation_text))

        score = base_score * 10 + overlap * 8 + strength

        scored.append({
            "score": score,
            "source": src,
            "target": tgt,
            "description": desc,
            "evidence": ev,
            "strength": strength,
            "chunk_id": data.get("chunk_id", ""),
            "title": title
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:max_relations]


# ============================================================
# 6. Build graph context
# ============================================================

def build_explicit_graph_context(question, max_relations=3):
    graph_results = graph_retrieve_explicit_with_chunk_context(
        question,
        top_k_entities=5,
        max_relations=max_relations,
        expand_chunks=True
    )

    if not graph_results:
        return ""

    parts = ["Graph-retrieved evidence:"]

    for item in graph_results:
        parts.append(
            f"Relation: {item['source']} -> {item['target']}\n"
            f"Description: {item['description']}\n"
            f"Evidence: {item['evidence']}\n"
            f"Strength: {item['strength']}\n"
            f"Source title: {item['title']}"
        )

    return "\n\n".join(parts)


# ============================================================
# 7. Graph-only answer
# ============================================================

def graph_only_rag_answer_explicit(question, max_relations=3):
    graph_context = build_explicit_graph_context(
        question,
        max_relations=max_relations
    )

    if not graph_context:
        graph_context = "No graph evidence found."

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{graph_context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    answer = gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return answer, graph_context


# ============================================================
# 8. True hybrid answer: DPR + explicit graph
# ============================================================

def true_hybrid_rag_answer_explicit(question, top_k=5, max_relations=3):
    docs = retrieve(question, top_k=top_k)

    dpr_context = "\n\n".join([
        f"Title: {d['title']}\nPassage: {d['text']}"
        for d in docs[:top_k]
    ])

    graph_context = build_explicit_graph_context(
        question,
        max_relations=max_relations
    )

    hybrid_context = dpr_context

    if graph_context:
        hybrid_context += "\n\nAdditional graph-retrieved evidence:\n"
        hybrid_context += graph_context

    prompt = f"""
Answer the question using only the context below.
Return only the short answer.

Question: {question}

Context:
{hybrid_context}

Answer:
"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=4,
            early_stopping=True
        )

    answer = gen_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return answer, hybrid_context, docs, graph_context


# ============================================================
# 9. Evaluation
# ============================================================

def evaluate_true_hybrid_explicit_graph(
    benchmark_df,
    limit=100,
    top_k=5,
    max_relations=3
):
    results = []
    subset = benchmark_df.head(limit)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        q = row["question"]
        gold = row["answers"]

        try:
            docs = retrieve(q, top_k=top_k)

            baseline_pred = generate_with_flan_t5(
                q,
                docs,
                top_k=top_k
            )

            graph_pred, graph_context = graph_only_rag_answer_explicit(
                q,
                max_relations=max_relations
            )

            hybrid_pred, hybrid_context, _, graph_context = true_hybrid_rag_answer_explicit(
                q,
                top_k=top_k,
                max_relations=max_relations
            )

            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": baseline_pred,
                "graph_pred": graph_pred,
                "true_hybrid_pred": hybrid_pred,
                "graph_context": graph_context,
                "graph_evidence_chars": len(graph_context),
                "baseline_EM": exact_match_score(baseline_pred, gold),
                "graph_EM": exact_match_score(graph_pred, gold),
                "true_hybrid_EM": exact_match_score(hybrid_pred, gold),
                "baseline_F1": f1_multi(baseline_pred, gold),
                "graph_F1": f1_multi(graph_pred, gold),
                "true_hybrid_F1": f1_multi(hybrid_pred, gold),
                "status": "ok"
            })

        except Exception as e:
            results.append({
                "question": q,
                "answers": gold,
                "baseline_pred": "",
                "graph_pred": "",
                "true_hybrid_pred": "",
                "graph_context": "",
                "graph_evidence_chars": 0,
                "baseline_EM": 0.0,
                "graph_EM": 0.0,
                "true_hybrid_EM": 0.0,
                "baseline_F1": 0.0,
                "graph_F1": 0.0,
                "true_hybrid_F1": 0.0,
                "status": str(e)
            })

    result_df = pd.DataFrame(results)

    print("========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========")
    print("Examples:", len(result_df))
    print("Top-k:", top_k)
    print("Max graph relations:", max_relations)

    print("\nBaseline EM:", result_df["baseline_EM"].mean() * 100)
    print("Graph-only EM:", result_df["graph_EM"].mean() * 100)
    print("True hybrid EM:", result_df["true_hybrid_EM"].mean() * 100)

    print("\nBaseline F1:", result_df["baseline_F1"].mean() * 100)
    print("Graph-only F1:", result_df["graph_F1"].mean() * 100)
    print("True hybrid F1:", result_df["true_hybrid_F1"].mean() * 100)

    print("\nGraph evidence available:",
          (result_df["graph_evidence_chars"] > 0).mean() * 100)

    return result_df




Explicit graph built.
Nodes: 117447
Edges: 151256
Chunks with graph edges: 17221


100%|██████████| 100/100 [08:04<00:00,  4.85s/it]

========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 3

Baseline EM: 4.0
Graph-only EM: 6.0
True hybrid EM: 6.0

Baseline F1: 8.466666666666667
Graph-only F1: 10.616666666666667
True hybrid F1: 11.697835497835497

Graph evidence available: 100.0


In [14]:
explicit_graph_chunk_test_100 = evaluate_true_hybrid_explicit_graph(
    nq_df,
    limit=100,
    top_k=5,
    max_relations=2
)

explicit_graph_chunk_test_100 = evaluate_true_hybrid_explicit_graph(
    nq_df,
    limit=100,
    top_k=5,
    max_relations=5
)


100%|██████████| 100/100 [08:15<00:00,  4.96s/it]


========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 2

Baseline EM: 4.0
Graph-only EM: 4.0
True hybrid EM: 6.0

Baseline F1: 8.466666666666667
Graph-only F1: 9.324603174603174
True hybrid F1: 10.135714285714286

Graph evidence available: 100.0


100%|██████████| 100/100 [08:06<00:00,  4.87s/it]

========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========
Examples: 100
Top-k: 5
Max graph relations: 5

Baseline EM: 4.0
Graph-only EM: 5.0
True hybrid EM: 6.0

Baseline F1: 8.466666666666667
Graph-only F1: 9.519047619047619
True hybrid F1: 12.616666666666667

Graph evidence available: 100.0


In [15]:
explicit_graph_chunk_full_r5 = evaluate_true_hybrid_explicit_graph(
    nq_df,
    limit=1000,
    top_k=5,
    max_relations=3
)

100%|██████████| 1000/1000 [1:08:51<00:00,  4.13s/it]

========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========
Examples: 1000
Top-k: 5
Max graph relations: 3

Baseline EM: 5.5
Graph-only EM: 5.3
True hybrid EM: 6.5

Baseline F1: 10.50123839123839
Graph-only F1: 9.66563492063492
True hybrid F1: 11.487683231800878

Graph evidence available: 100.0


In [16]:
explicit_graph_chunk_full_r5 = evaluate_true_hybrid_explicit_graph(
    nq_df,
    limit=len(nq_df),
    top_k=5,
    max_relations=3
)

100%|██████████| 3610/3610 [5:08:13<00:00,  5.12s/it]  

========== EXPLICIT GRAPH + CHUNK CONTEXT RESULTS ==========
Examples: 3610
Top-k: 5
Max graph relations: 3

Baseline EM: 5.8171745152354575
Graph-only EM: 4.6814404432132966
True hybrid EM: 6.260387811634349

Baseline F1: 10.582810428446436
Graph-only F1: 8.354055346641301
True hybrid F1: 10.944373516151257

Graph evidence available: 99.97229916897507
